# Lab 3 — Interrogate Your Photos
**Session 3 · Multimodal · TCE** · First: **File → Save a copy in Drive**.

You need 2–3 photos: anything on your phone — a receipt, your handwritten notes, the canteen menu board. Transfer to laptop (email yourself / Drive / USB cable).

In [ ]:
# Cell 1 — setup
%pip install -q -U google-genai pillow
from getpass import getpass
from google import genai
from google.genai import types
import logging; logging.getLogger("google_genai").setLevel(logging.ERROR)   # hide harmless SDK notices (e.g. "non-text parts: thought_signature")
from PIL import Image
import time

MOCK = False   # ← set True only if the instructor says the live API is unavailable

client = genai.Client(api_key="mock" if MOCK else getpass("Gemini API key: "))
MODEL = "gemini-flash-lite-latest"  # lite: minimal thinking by default (near-zero thinking tokens on simple prompts), so the free tier goes further (Aug 2026 → Gemini 3.5 Flash Lite). 429 = rate limit: the helper below waits and retries; 503 = high demand: wait and re-run.

def _shrink(x, cap=1024):
    """Downscale big images before sending — saves free-tier quota and time.
    A 4000px phone photo and a 1024px one give the model the same answer."""
    if isinstance(x, Image.Image) and max(x.size) > cap:
        r = cap / max(x.size)
        return x.resize((int(x.width*r), int(x.height*r)))
    return x

def open_image(path):
    """Image.open — but under MOCK a missing file becomes a blank placeholder, so Run-all still works."""
    try:
        return Image.open(path)
    except FileNotFoundError:
        if MOCK:
            print(f"[MOCK] {path} not found — using a blank placeholder image")
            return Image.new("RGB", (400, 300), (235, 235, 235))
        raise

_MOCK_ANSWERS = [   # (keyword, canned reply) — offline path only; describes an imaginary Madurai receipt
    ("reply exactly: unreadable", "UNREADABLE"),
    ("extract",       '{"vendor": "Murugan Idli Shop", "date": "2026-09-19", "items": [{"name": "Idli set", "price": 80.0}, {"name": "Ghee pongal", "price": 95.0}, {"name": "Medu vada", "price": 45.0}, {"name": "Masala dosa", "price": 92.0}, {"name": "Filter coffee", "price": 30.0}], "total": 342.0}'),
    ("total",         "The total on the receipt is ₹342.00."),
    ("describe",      "A printed till receipt from a small South Indian restaurant, lying on a wooden table. Five line items are listed with a total at the bottom."),
    ("read all text", "MURUGAN IDLI SHOP\nAnna Nagar, Madurai\nIdli set 80.00\nGhee pongal 95.00\nMedu vada 45.00\nMasala dosa 92.00\nFilter coffee 30.00\nTOTAL 342.00\nThank you, visit again"),
    ("how many",      "There are 3 distinct objects: the receipt, a pen, and the edge of a coffee cup."),
    ("infer",         "A South Indian restaurant, probably Madurai (the header says Anna Nagar); a weekday morning, given the breakfast items."),
    ("surprising",    "The receipt has no GST number printed, which is unusual for a printed bill."),
    ("transcribe",    "Operating Systems — Unit 3\nDeadlock: mutual exclusion, hold & wait, no pre-emption, circular [?]\nBanker's algorithm → safe state [?] sequence\nRevise: resource allocation graph"),
]
_MOCK_DEFAULT = "I can see the image, but that question needs more detail."

def ask(contents, temperature=None):
    """contents can be a string, or a list mixing PIL Images and strings."""
    if MOCK:
        p = " ".join(c for c in (contents if isinstance(contents, list) else [contents]) if isinstance(c, str)).lower()
        return "[MOCK] " + next((a for k, a in _MOCK_ANSWERS if k in p), _MOCK_DEFAULT)
    if isinstance(contents, list):
        contents = [_shrink(c) for c in contents]
    config = types.GenerateContentConfig(temperature=temperature) if temperature is not None else None
    for attempt in range(4):
        try:
            # blocked/empty responses come back as None — return "" so .strip()/json.loads fail loudly, not with a TypeError
            return client.models.generate_content(model=MODEL, contents=contents, config=config).text or ""
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited..."); time.sleep(20*(attempt+1))
            else: raise
print("ready ✓" + ("  (MOCK mode — canned answers about an imaginary receipt)" if MOCK else ""))

import re, textwrap
def wrap(text, width=80):
    """Word-wrap a reply so it fits the screen. Keeps the model's own line breaks, bullets and code blocks."""
    out, in_code = [], False
    for line in str(text).splitlines():
        if line.lstrip().startswith("```"):
            in_code = not in_code
        if in_code or line.lstrip().startswith("```") or not line.strip():
            out.append(line); continue
        m = re.match(r"\s*(?:[-*•]|\d+[.)])\s+", line)          # bullet / numbered item → hang-indent
        hang = " " * len(m.group(0)) if m else line[:len(line) - len(line.lstrip())]
        out.append(textwrap.fill(line, width, subsequent_indent=hang,
                                 break_long_words=False, break_on_hyphens=False))
    return "\n".join(out)

## Part A — Photo Q&A (escalating difficulty)

Upload a photo: **folder icon (left sidebar) → upload**. Then run the ladder.

No photo handy? Three verified public-domain sample images (Wikimedia Commons):
- Receipt: https://commons.wikimedia.org/wiki/Special:FilePath/ReceiptSwiss.jpg
- Madurai Meenakshi temple: https://commons.wikimedia.org/wiki/Special:FilePath/Madurai_Meenakshi_Amman_Temple.jpg
- Handwriting (for Part C): Abraham Lincoln's 1854 letter to Mr. Brayman, US National Archives — https://upload.wikimedia.org/wikipedia/commons/thumb/4/49/Abraham_Lincoln_Letter_to_Mr._Brayman_September_23%2C_1854_-_NARA_-_192847.jpg/1280px-Abraham_Lincoln_Letter_to_Mr._Brayman_September_23%2C_1854_-_NARA_-_192847.jpg

Un-comment the `wget` line in the next cell to pull one straight into Colab.

In [ ]:
# Cell 2 — the interrogation ladder
# No photo of your own? Upload sample-inputs/sample-photo.jpg from the lab folder (folder icon,
# left sidebar) and put that filename below. Ground truth for counting is in the folder's README.
# Or un-comment the next line to fetch one from the web, then use "sample.jpg":
# !wget -q "https://commons.wikimedia.org/wiki/Special:FilePath/ReceiptSwiss.jpg" -O sample.jpg
# (the handwriting sample for Part C is in Cell 5)
img = open_image("your_photo.jpg")   # ← your filename
display(img.resize((min(400, img.width), int(img.height * min(400, img.width) / img.width))))

questions = [
    "Describe this image in 2 sentences.",
    "Read ALL text visible in this image, exactly as written.",
    "How many distinct objects/people are in this image? Count carefully.",
    "What can you infer about where and when this was taken?",
    "What is the most surprising detail in this image?",
]
for i, q in enumerate(questions, 1):
    print("=" * 70)
    print(f"Q{i}. {q}")
    print("-" * 70)
    print(wrap(ask([img, q])), "\n")
# Which answers can you check against the photo? Which ones did it make up (Q3 count, Q4 guesses)?

### Grade it (edit this cell)
Which answers were right? Where did it wobble — counting? small text? inference?

### ✓ Checkpoint 1 — ladder run + your 2-line grading.

---
## Part B — Receipt / document → JSON

Session 2's format control, now with eyes. Strict schema, `ONLY`, grounding line.

In [ ]:
# Cell 3 — structured extraction
doc = open_image("receipt.jpg")   # ← receipt / bill / marksheet / form
# No bill handy? Upload sample-inputs/sample-receipt.jpg (5 items, total 342.00) or
# sample-inputs/sample-marksheet.jpg (five subjects, total 195 — one mark is faint on purpose).

schema_prompt = """Extract data from this image.
Reply ONLY with JSON in exactly this schema:
{"vendor": str, "date": str, "items": [{"name": str, "price": float}], "total": float}
If any field is unreadable, use null — do NOT guess."""

reply = ask([doc, schema_prompt], temperature=0.0)
print("The model's raw reply — to your program this is just TEXT until Cell 4 parses it:\n")
print(wrap(reply))   # wrapped for display only

In [ ]:
# Cell 4 — prove it parses (the real test)
import json as pyjson
raw = ask([doc, schema_prompt], temperature=0.0)
# strip accidental code fences if present
raw = raw.strip().removeprefix("[MOCK] ").removeprefix("```json").removeprefix("```").removesuffix("```").strip()
data = pyjson.loads(raw)                      # ← the real test: crashes if the reply is not valid JSON

def show_receipt(d):
    """Print the extracted fields as a readable bill, and check the arithmetic."""
    money = lambda v: f"{v:>9.2f}" if isinstance(v, (int, float)) else "  (null)"
    print(f"  vendor : {d.get('vendor')}")
    print(f"  date   : {d.get('date')}")
    items = d.get("items") or []
    print(f"  items  : {len(items)}")
    for it in items:
        print(f"     {str(it.get('name')):<28}{money(it.get('price'))}")
    total = d.get("total")
    print(f"  total  : {'':<22}{money(total)}")
    prices = [it.get("price") for it in items if isinstance(it.get("price"), (int, float))]
    if isinstance(total, (int, float)) and prices:
        s = sum(prices)
        if abs(s - total) < 0.01:
            print(f"\n  check  : items add up to {s:.2f} = total ✓")
        else:
            print(f"\n  check  : items add up to {s:.2f} but total says {total:.2f} ✗  (a misread price, a missing item, or tax)")
    else:
        print("\n  check  : cannot add up — some prices or the total are null (unreadable)")

print("PARSED ✓ — the reply is valid JSON. What the model extracted:\n")
show_receipt(data)
print("\nNow hold the photo next to this list: every value must match the image.")
# If this cell crashes, your prompt isn't strict enough. Tighten and re-run — that's the lesson.

### ✓ Checkpoint 2 — `json.loads` succeeds on your document.

---
## Part B2 — the production way: constrained structure

Prompt instructions request a format. A response schema constrains the **shape** of the output and reduces formatting failures, but it does not guarantee correct values, a successful request, or a useful answer when the image is unreadable. The schema below allows nullable fields so the model has an honest way to say that a value is unknown.

### ✓ Checkpoint 3 — the structured response parses, and you inspect whether its values are supported by the image.

In [ ]:
# Cell 4b — response_schema: constrain shape, then verify values
schema = {
    "type": "OBJECT",
    "properties": {
        "vendor": {"type": "STRING", "nullable": True},
        "date":   {"type": "STRING", "nullable": True},
        "total":  {"type": "NUMBER", "nullable": True},
        "items":  {"type": "ARRAY", "nullable": True, "items": {
            "type": "OBJECT",
            "properties": {
                "name": {"type": "STRING", "nullable": True},
                "price": {"type": "NUMBER", "nullable": True},
            },
            "required": ["name", "price"],
        }},
    },
    "required": ["vendor", "date", "total", "items"],
}

if MOCK:
    raw = ask([doc, "Extract the receipt."]).removeprefix("[MOCK] ")   # canned JSON, so the cell runs offline
else:
    resp = client.models.generate_content(
        model=MODEL,
        contents=[_shrink(doc), "Extract the receipt. Use null for any field that is not clearly readable."],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    raw = resp.text or "{}"      # blocked → empty object → every field shows as None below: the honest failure
data = pyjson.loads(raw)
print("PARSED ✓ — the schema guaranteed the SHAPE. Now check the VALUES:\n")
show_receipt(data)             # from Cell 4
print("\nValid JSON is not evidence that the extraction is correct — compare every value with the image.")

---
## Part C — Handwriting test

Photograph a page of YOUR handwritten notes → transcribe → grade yourself: roughly what % correct? Tamil/Tanglish notes = bonus experiment.

In [ ]:
# Cell 5 — handwriting
# No page of your own? Un-comment the wget: Abraham Lincoln's letter to Mr. Brayman, 23 Sep 1854 (US National
# Archives, public domain; 1280px Wikimedia thumbnail, verified Sep 2026). 19th-century cursive — a fair test.
# !wget -q "https://upload.wikimedia.org/wikipedia/commons/thumb/4/49/Abraham_Lincoln_Letter_to_Mr._Brayman_September_23%2C_1854_-_NARA_-_192847.jpg/1280px-Abraham_Lincoln_Letter_to_Mr._Brayman_September_23%2C_1854_-_NARA_-_192847.jpg" -O my_notes.jpg
# No handwriting handy? Upload sample-inputs/sample-handwriting.jpg — the README has what it says.
notes = open_image("my_notes.jpg")
print(ask([notes, "Transcribe this handwritten page exactly. Mark unclear words as [?]."]))

## Part D — Break it

Find one image where the model **confidently invents a detail**: a blurred price it "reads" anyway, objects it miscounts, text it paraphrases instead of quoting.

### ✓ Checkpoint 4 — show me the invention.

---
## Stretch goals

In [ ]:
# Stretch 1 — does grounding stop the invention?
# Re-run your Part D image WITH the grounding line vs WITHOUT:
loose  = "What is the total on this receipt?"
strict = "What is the total on this receipt? If it is not clearly readable, reply exactly: UNREADABLE."
img_d = open_image("your_partD_image.jpg")   # sample-inputs/sample-marksheet.jpg works here too
a_loose, a_strict = ask([img_d, loose]), ask([img_d, strict])
print(wrap("loose  (no escape hatch) : " + a_loose))
print(wrap("strict (escape hatch)    : " + a_strict))
print()
print(wrap("How to read this: same number in both → the image is readable. A confident number from "
           "'loose' but UNREADABLE from 'strict' → the loose number was a guess the model dressed up as a fact."))

In [ ]:
# Stretch 2 — vision eval (your S2 harness grows eyes)
# this becomes 25% of your capstone score — aim for 10 examples, not 5
vision_tests = [
    {"img": "receipt.jpg", "q": "What is the total?", "expected": "342"},
    # add 4 more: photo, question, expected key fact
]
import re
def norm(s): return re.sub(r"[^a-z0-9 ]", "", s.lower())
hits = 0
for t in vision_tests:
    ans = ask([open_image(t["img"]), t["q"]], temperature=0.0)
    ok = norm(t["expected"]) in norm(ans); hits += ok
    print(("✓" if ok else "✗"), f"{t['img']} · {t['q']}")
    print("   expected:", t["expected"])
    print(textwrap.fill(" ".join(ans.split())[:240], 80, initial_indent="   got:      ", subsequent_indent=" " * 13))
    print()
print(f"VISION SCORE: {hits}/{len(vision_tests)}   (n={len(vision_tests)} — add more photos before you trust this number)")

In [ ]:
# Stretch 3 — audio: record a short voice note on your phone, upload it
if MOCK:
    print("[MOCK] audio needs the live API — skipped")
else:
    # No recording yet? sample-inputs/sample-voice-note.m4a is a 20-second synthetic note.
    audio = client.files.upload(file="voicenote.m4a")
    print(ask([audio, "Transcribe this audio, then summarize it in one line."]))

## Day 1 complete

**Tonight (5 min, mandatory):** put 2–3 real documents (lecture notes, textbook chapter PDF) on your laptop/Drive. Tomorrow: **chat with your notes** — the pattern behind most real AI products.

Sleep well. Day 2 is the good stuff.